## Import essential libraries to use throughout the project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')

In [ ]:
df.sample(5)

In [ ]:
df.shape

## Data Cleaning

In [ ]:
df.info()

In [ ]:
# Drop column 2, 3, 4 since they have very little values that are not NaN.

df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

In [ ]:
# Rename the column names

df.rename(columns={'v1': 'target', 'v2':'text_input'}, inplace=True)

In [ ]:
df

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [ ]:
df['target'] = encoder.fit_transform(df['target'])

In [ ]:
df.head()

In [ ]:
# Check missing values

df.isnull().sum()

In [ ]:
# Check duplicate values

df.duplicated().sum()

In [ ]:
# Remove Duplicates

df = df.drop_duplicates(keep='first')

In [ ]:
# Check duplicate values

df.duplicated().sum()

In [ ]:
df.shape

## EDA

In [ ]:
df['target'].value_counts()

In [ ]:
plt.pie(df['target'].value_counts(), labels=['ham', 'spam'], autopct="%0.2f")

In [ ]:
import ssl
import nltk

ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Get number of characters in each message

df['char_count'] = df['text_input'].str.len()

In [ ]:
df.head()

In [ ]:
# Get number of words in each message

df['word_count'] = df['text_input'].str.split().str.len()

In [ ]:
df.head()

In [ ]:
# Get number of sentences in each message

df['sentence_count'] = df['text_input'].apply(lambda x: len(nltk.sent_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df[['char_count', 'word_count', 'sentence_count']].describe()

## Import essential libraries to use throughout the project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')

In [ ]:
df.sample(5)

In [ ]:
df.shape

## Data Cleaning

In [ ]:
df.info()

In [ ]:
# Drop column 2, 3, 4 since they have very little values that are not NaN.

df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

In [ ]:
# Rename the column names

df.rename(columns={'v1': 'target', 'v2':'text_input'}, inplace=True)

In [ ]:
df

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [ ]:
df['target'] = encoder.fit_transform(df['target'])

In [ ]:
df.head()

In [ ]:
# Check missing values

df.isnull().sum()

In [ ]:
# Check duplicate values

df.duplicated().sum()

In [ ]:
# Remove Duplicates

df = df.drop_duplicates(keep='first')

In [ ]:
# Check duplicate values

df.duplicated().sum()

In [ ]:
df.shape

## EDA

In [ ]:
df['target'].value_counts()

In [ ]:
plt.pie(df['target'].value_counts(), labels=['ham', 'spam'], autopct="%0.2f")

In [ ]:
import ssl
import nltk

ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Get number of characters in each message

df['char_count'] = df['text_input'].str.len()

In [ ]:
df.head()

In [ ]:
# Get number of words in each message

df['word_count'] = df['text_input'].str.split().str.len()

In [ ]:
df.head()

In [ ]:
# Get number of sentences in each message

df['sentence_count'] = df['text_input'].apply(lambda x: len(nltk.sent_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df[['char_count', 'word_count', 'sentence_count']].describe()

In [ ]:
# Describe output message for 'ham' messages

df[df['target'] == 0][['char_count', 'word_count', 'sentence_count']].describe()

In [ ]:
# Describe output message for 'spam' messages

df[df['target'] == 1][['char_count', 'word_count', 'sentence_count']].describe()

In [ ]:
sns.histplot(df[df['target'] == 0]['char_count'])
sns.histplot(df[df['target'] == 1]['char_count'], color='red')

In [ ]:
sns.histplot(df[df['target'] == 0]['word_count'])
sns.histplot(df[df['target'] == 1]['word_count'], color='red')

In [ ]:
sns.histplot(df[df['target'] == 0]['sentence_count'])
sns.histplot(df[df['target'] == 1]['sentence_count'], color='red')

In [ ]:
sns.pairplot(df, hue='target')

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

## Preprocessing

* Lower case
* Tokenization
* Remove special chars
* Removing stop words and punctuation
* Stemming

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
import string

In [ ]:
def transform_text(text):
    text = text.lower()
    text = nltk.word_tokenize(text)

    new_text = []
    for i in text:
        if i.isalnum():
            new_text.append(i)

    text = new_text[:]
    new_text.clear()

    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            new_text.append(i)

    text = new_text[:]
    new_text.clear()

    for i in text:
        new_text.append(ps.stem(i))

    return " ".join(new_text)

In [ ]:
df['transformed_text'] = df['text_input'].apply(transform_text)

In [ ]:
df.sample(5)

In [ ]:
from collections import Counter

spam_words = []

for message in df[df['target'] == 1]['transformed_text'].tolist():
    spam_words.extend(message.split())

word_counts = Counter(spam_words)

print(word_counts.most_common(30))

## Build Model

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
cv = CountVectorizer()
tfidf = TfidfVectorizer(max_features=3000)

In [ ]:
X = cv.fit_transform(df['transformed_text']).toarray()
X1 = tfidf.fit_transform(df['transformed_text']).toarray()

In [ ]:
X1.shape

In [ ]:
y = df['target'].values

In [ ]:
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y, test_size=0.2, random_state=0)

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score

gnb = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()

In [ ]:
gnb.fit(X_train, y_train)
ypred1 = gnb.predict(X_test)

print(accuracy_score(y_test, ypred1)*100)
print(confusion_matrix(y_test, ypred1))
print(precision_score(y_test, ypred1)*100)

In [ ]:
gnb.fit(X1_train, y_train)
ypred1 = gnb.predict(X1_test)

print(accuracy_score(y_test, ypred1)*100)
print(confusion_matrix(y_test, ypred1))
print(precision_score(y_test, ypred1)*100)

In [ ]:
mnb.fit(X_train, y_train)
ypred2 = mnb.predict(X_test)

print(accuracy_score(y_test, ypred2)*100)
print(confusion_matrix(y_test, ypred2))
print(precision_score(y_test, ypred2)*100)

In [ ]:
mnb.fit(X1_train, y_train)
ypred2 = mnb.predict(X1_test)

print(accuracy_score(y_test, ypred2)*100)
print(confusion_matrix(y_test, ypred2))
print(precision_score(y_test, ypred2)*100)

In [ ]:
bnb.fit(X_train, y_train)
ypred3 = bnb.predict(X_test)

print(accuracy_score(y_test, ypred3)*100)
print(confusion_matrix(y_test, ypred3))
print(precision_score(y_test, ypred3)*100)

In [ ]:
bnb.fit(X1_train, y_train)
ypred3 = bnb.predict(X1_test)

print(accuracy_score(y_test, ypred3)*100)
print(confusion_matrix(y_test, ypred3))
print(precision_score(y_test, ypred3)*100)

## Export model for deployment

In [ ]:
import pickle
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(mnb, open('model.pkl', 'wb'))